# Load and Inspect

In [1]:
import pandas as pd
import numpy as np

In [14]:
permits = pd.read_csv("permits_by_tract_year.csv")
train   = pd.read_csv("merged_training_data_long.csv")
val     = pd.read_csv("merged_validation_data_long.csv")
test    = pd.read_csv("merged_testing_data.csv")

# Standardize tract_id to 11-digit string across all files
for df in [permits, train, val, test]:
    df["tract_id"] = df["tract_id"].astype(str).str.zfill(11)

# Valuation of 0.0 means no data for SF/Oakland — treat as NaN
permits["new_const_valuation_sum"]   = permits["new_const_valuation_sum"].replace(0.0, np.nan)
permits["all_permits_valuation_sum"] = permits["all_permits_valuation_sum"].replace(0.0, np.nan)

# Map county FIPS to city name for within-city normalization
county_to_city = {
    "06075": "San Francisco",
    "06001": "Oakland/Alameda",
    "06013": "Contra Costa",
    "06081": "San Mateo",
    "06085": "San Jose",
}
permits["city"] = permits["tract_id"].str[:5].map(county_to_city).fillna("Other")

print("Permits:", permits.shape, "| Years:", permits["year_filed"].min(), "–", permits["year_filed"].max())
print("Train:", train.shape)
print("Val:  ", val.shape)
print("Test: ", test.shape)
print("\nPermit counts by city:")
print(permits.groupby("city")["new_const_count"].sum().sort_values(ascending=False))

Permits: (3782, 13) | Years: 2004 – 2026
Train: (1744, 35)
Val:   (872, 35)
Test:  (867, 31)

Permit counts by city:
city
San Jose           7794.0
San Francisco      2439.0
Oakland/Alameda    1614.0
Contra Costa          1.0
Name: new_const_count, dtype: float64


# Aggregate permits into 5-year windows

In [15]:
PERMIT_COLS = [
    "new_const_count",
    "new_const_valuation_sum",
    "new_const_sqft_sum",
    "new_const_units_sum",
    "all_permits_count",
]

def aggregate_window(permits_df, year_start, year_end, suffix):
    window = permits_df[permits_df["year_filed"].between(year_start, year_end)]
    agg = (window
           .groupby("tract_id")[PERMIT_COLS]
           .sum()
           .reset_index())
    agg = agg.rename(columns={c: f"{c}_{suffix}" for c in PERMIT_COLS})
    return agg

# 5-year window centered on each feature snapshot
permits_train = aggregate_window(permits, 2006, 2010, "train")
permits_val   = aggregate_window(permits, 2011, 2015, "val")
permits_test  = aggregate_window(permits, 2020, 2024, "test")

print("Training window (2008–2012):", permits_train.shape)
print(permits_train[["new_const_count_train","new_const_sqft_sum_train","new_const_units_sum_train"]].describe().round(1))

Training window (2008–2012): (364, 6)
       new_const_count_train  new_const_sqft_sum_train  \
count                  364.0                     364.0   
mean                     7.5                   44011.5   
std                     16.7                  136064.5   
min                      0.0                       0.0   
25%                      1.0                       0.0   
50%                      3.0                    1152.5   
75%                      6.0                   14037.8   
max                    177.0                 1165084.0   

       new_const_units_sum_train  
count                      364.0  
mean                        17.3  
std                         60.5  
min                          0.0  
25%                          0.0  
50%                          0.0  
75%                          2.0  
max                        598.0  


# Merge into training set

In [16]:
train_merged = train.merge(permits_train, on="tract_id", how="left")

# Tracts with no permits → 0 (no activity, not missing)
permit_train_cols = [c for c in train_merged.columns if c.endswith("_train")]
train_merged[permit_train_cols] = train_merged[permit_train_cols].fillna(0)

# Within-city percentile rank to correct for San Jose data density
train_merged["city_group"] = train_merged["tract_id"].str[:5].map(county_to_city).fillna("Other")
for col in ["new_const_count_train", "new_const_sqft_sum_train", "new_const_units_sum_train"]:
    train_merged[f"{col}_pct"] = train_merged.groupby("city_group")[col].rank(pct=True)

print("Training set after merge:", train_merged.shape)
print("Tracts with permit activity:", (train_merged["new_const_count_train"] > 0).sum(), "of", len(train_merged))
print("\nNew columns:")
print([c for c in train_merged.columns if "new_const" in c or "all_permits" in c])

Training set after merge: (1744, 44)
Tracts with permit activity: 494 of 1744

New columns:
['new_const_count_train', 'new_const_valuation_sum_train', 'new_const_sqft_sum_train', 'new_const_units_sum_train', 'all_permits_count_train', 'new_const_count_train_pct', 'new_const_sqft_sum_train_pct', 'new_const_units_sum_train_pct']


# Merge into validation set

In [17]:
val_merged = val.merge(permits_val, on="tract_id", how="left")

permit_val_cols = [c for c in val_merged.columns if c.endswith("_val")]
val_merged[permit_val_cols] = val_merged[permit_val_cols].fillna(0)

val_merged["city_group"] = val_merged["tract_id"].str[:5].map(county_to_city).fillna("Other")
for col in ["new_const_count_val", "new_const_sqft_sum_val", "new_const_units_sum_val"]:
    val_merged[f"{col}_pct"] = val_merged.groupby("city_group")[col].rank(pct=True)

print("Validation set after merge:", val_merged.shape)
print("Tracts with permit activity:", (val_merged["new_const_count_val"] > 0).sum(), "of", len(val_merged))

Validation set after merge: (872, 44)
Tracts with permit activity: 229 of 872


# Merge into test set

In [18]:
test_merged = test.merge(permits_test, on="tract_id", how="left")

permit_test_cols = [c for c in test_merged.columns if c.endswith("_test")]
test_merged[permit_test_cols] = test_merged[permit_test_cols].fillna(0)

test_merged["city_group"] = test_merged["tract_id"].str[:5].map(county_to_city).fillna("Other")
for col in ["new_const_count_test", "new_const_sqft_sum_test", "new_const_units_sum_test"]:
    test_merged[f"{col}_pct"] = test_merged.groupby("city_group")[col].rank(pct=True)

print("Test set after merge:", test_merged.shape)
print("Tracts with permit activity:", (test_merged["new_const_count_test"] > 0).sum(), "of", len(test_merged))

Test set after merge: (867, 40)
Tracts with permit activity: 242 of 867


# Sanity check and save

In [19]:
print("=== Final shapes ===")
print(f"  Training   : {train_merged.shape}")
print(f"  Validation : {val_merged.shape}")
print(f"  Test       : {test_merged.shape}")

print("\n=== Permit coverage by city ===")
for label, df, col in [
    ("Train", train_merged, "new_const_count_train"),
    ("Val",   val_merged,   "new_const_count_val"),
    ("Test",  test_merged,  "new_const_count_test"),
]:
    coverage = (df[col] > 0).groupby(df["city_group"]).mean().round(2)
    print(f"\n{label}:")
    print(coverage.to_string())


train_merged.to_csv("training_with_permits.csv", index=False)
val_merged.to_csv("validation_with_permits.csv", index=False)
test_merged.to_csv("test_with_permits.csv", index=False)
print("\nSaved: training_with_permits.csv, validation_with_permits.csv, test_with_permits.csv")

=== Final shapes ===
  Training   : (1744, 44)
  Validation : (872, 44)
  Test       : (867, 40)

=== Permit coverage by city ===

Train:
city_group
Contra Costa       0.00
Oakland/Alameda    0.29
San Francisco      0.75
San Jose           0.35
San Mateo          0.00

Val:
city_group
Contra Costa       0.00
Oakland/Alameda    0.23
San Francisco      0.75
San Jose           0.34
San Mateo          0.00

Test:
city_group
Contra Costa       0.00
Oakland/Alameda    0.10
San Francisco      0.72
San Jose           0.53
San Mateo          0.00

=== Leakage check ===
Training permit window : 2008–2012  → predicts 2010→2015 change ✓
Validation permit window: 2013–2017 → predicts 2015→2020 change ✓
Test permit window      : 2021–2024 → predicts future change     ✓

Saved: training_with_permits.csv, validation_with_permits.csv, test_with_permits.csv


In [21]:
from google.colab import files

files.download("training_with_permits.csv")
files.download("validation_with_permits.csv")
files.download("test_with_permits.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Refining Permit Columns

In [22]:
from google.colab import files
import pandas as pd
import numpy as np

uploaded = files.upload()  # upload all three at once

train = pd.read_csv("training_with_permits.csv")
val   = pd.read_csv("validation_with_permits.csv")
test  = pd.read_csv("test_with_permits.csv")

print("Train:", train.shape)
print("Val:  ", val.shape)
print("Test: ", test.shape)

Train: (1744, 44)
Val:   (872, 44)
Test:  (867, 40)


# Clean and Engineer Features

In [23]:
def process_permits(df, suffix):
    # Drop broken percentile columns
    pct_cols = [c for c in df.columns if c.endswith("_pct")]
    df = df.drop(columns=pct_cols)

    # Compute per-permit scale — NaN where count is 0
    count = df[f"new_const_count_{suffix}"].replace(0, np.nan)

    df["avg_valuation_per_permit"] = df[f"new_const_valuation_sum_{suffix}"] / count
    df["avg_sqft_per_permit"]      = df[f"new_const_sqft_sum_{suffix}"]      / count
    df["avg_units_per_permit"]     = df[f"new_const_units_sum_{suffix}"]      / count
    df["any_permit_activity"]      = (df[f"new_const_count_{suffix}"] > 0).astype(int)

    # Zero-permit tracts get 0 for scale features (no investment = meaningful signal)
    df[["avg_valuation_per_permit",
        "avg_sqft_per_permit",
        "avg_units_per_permit"]] = df[["avg_valuation_per_permit",
                                       "avg_sqft_per_permit",
                                       "avg_units_per_permit"]].fillna(0)
    return df

train = process_permits(train, "train")
val   = process_permits(val,   "val")
test  = process_permits(test,  "test")

print("Train:", train.shape)
print("Val:  ", val.shape)
print("Test: ", test.shape)

new_cols = ["avg_valuation_per_permit", "avg_sqft_per_permit",
            "avg_units_per_permit", "any_permit_activity"]
print("\nNew feature summary (training):")
print(train[new_cols].describe().round(1))

Train: (1744, 45)
Val:   (872, 45)
Test:  (867, 41)

New feature summary (training):
       avg_valuation_per_permit  avg_sqft_per_permit  avg_units_per_permit  \
count                    1744.0               1744.0                1744.0   
mean                   120918.6               1280.4                   0.5   
std                   1111491.1               7724.6                   4.8   
min                         0.0                  0.0                   0.0   
25%                         0.0                  0.0                   0.0   
50%                         0.0                  0.0                   0.0   
75%                         0.0                  0.0                   0.0   
max                  25000000.0             139725.8                 110.5   

       any_permit_activity  
count               1744.0  
mean                   0.3  
std                    0.5  
min                    0.0  
25%                    0.0  
50%                    0.0  
75%      

# Check Separation on train and val

In [24]:
check_cols = [
    "new_const_count_train",
    "new_const_valuation_sum_train",
    "new_const_sqft_sum_train",
    "new_const_units_sum_train",
    "avg_valuation_per_permit",
    "avg_sqft_per_permit",
    "avg_units_per_permit",
    "any_permit_activity",
]

print("=== TRAINING SET: mean by gentrification label ===")
print(train.groupby("gentrified")[check_cols].mean().round(1).T.to_string())

print("\n=== TRAINING SET: correlation with gentrified ===")
for col in check_cols:
    r = train[col].corr(train["gentrified"])
    print(f"  {col:45s}  r = {r:.3f}")

# Repeat for validation using _val suffix
val_check = [c.replace("_train", "_val") for c in check_cols
             if f"{c.replace('_train','_val')}" in val.columns or
             c in ["avg_valuation_per_permit","avg_sqft_per_permit",
                   "avg_units_per_permit","any_permit_activity"]]

print("\n=== VALIDATION SET: mean by gentrification label ===")
val_permit_cols = [c for c in val.columns if any(k in c for k in
                  ["new_const","all_permits","avg_","any_permit"])]
print(val.groupby("gentrified")[val_permit_cols].mean().round(1).T.to_string())

=== TRAINING SET: mean by gentrification label ===
gentrified                            0          1
new_const_count_train               2.4        0.9
new_const_valuation_sum_train  748020.6  1926801.9
new_const_sqft_sum_train         9944.1    28072.4
new_const_units_sum_train           3.6       15.5
avg_valuation_per_permit       113255.2   322081.0
avg_sqft_per_permit              1137.8     5025.3
avg_units_per_permit                0.4        2.9
any_permit_activity                 0.3        0.1

=== TRAINING SET: correlation with gentrified ===
  new_const_count_train                          r = -0.025
  new_const_valuation_sum_train                  r = 0.039
  new_const_sqft_sum_train                       r = 0.056
  new_const_units_sum_train                      r = 0.082
  avg_valuation_per_permit                       r = 0.035
  avg_sqft_per_permit                            r = 0.095
  avg_units_per_permit                           r = 0.099
  any_permit_activity    

# Save all three

In [25]:
train.to_csv("training_final_v2.csv",   index=False)
val.to_csv("validation_final_v2.csv",   index=False)
test.to_csv("testing_final_v2.csv",     index=False)

print("Saved:")
print(f"  training_final_v2.csv   — {train.shape[0]} rows, {train.shape[1]} cols")
print(f"  validation_final_v2.csv — {val.shape[0]} rows, {val.shape[1]} cols")
print(f"  testing_final_v2.csv    — {test.shape[0]} rows, {test.shape[1]} cols")

print("\nFinal permit columns in each file:")
for name, df in [("Train", train), ("Val", val), ("Test", test)]:
    p = [c for c in df.columns if any(k in c for k in
         ["new_const","all_permits","avg_","any_permit"])]
    print(f"  {name}: {p}")

Saved:
  training_final_v2.csv   — 1744 rows, 45 cols
  validation_final_v2.csv — 872 rows, 45 cols
  testing_final_v2.csv    — 867 rows, 41 cols

Final permit columns in each file:
  Train: ['new_const_count_train', 'new_const_valuation_sum_train', 'new_const_sqft_sum_train', 'new_const_units_sum_train', 'all_permits_count_train', 'avg_valuation_per_permit', 'avg_sqft_per_permit', 'avg_units_per_permit', 'any_permit_activity']
  Val: ['new_const_count_val', 'new_const_valuation_sum_val', 'new_const_sqft_sum_val', 'new_const_units_sum_val', 'all_permits_count_val', 'avg_valuation_per_permit', 'avg_sqft_per_permit', 'avg_units_per_permit', 'any_permit_activity']
  Test: ['new_const_count_test', 'new_const_valuation_sum_test', 'new_const_sqft_sum_test', 'new_const_units_sum_test', 'all_permits_count_test', 'avg_valuation_per_permit', 'avg_sqft_per_permit', 'avg_units_per_permit', 'any_permit_activity']


In [26]:
from google.colab import files

files.download("training_final_v2.csv")
files.download("validation_final_v2.csv")
files.download("testing_final_v2.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>